# SentinelVoice — train anti-spoof on Colab / Kaggle GPU

Calls the **same** 	raining.train_antispoof / 	raining.calibrate entry points as local CLI.
Mount or upload ASVspoof under datasets/ (see scripts/fetch_datasets.md).
Do **not** tune thresholds on eval. Platt fits **DEV only**.

In [ ]:
# Optional: clone + install (edit URL if needed)
# !git clone https://github.com/YOUR_ORG/Sih_104.git
# %cd Sih_104/ml-engine
# !pip install -e .
from pathlib import Path
import sys
ROOT = Path.cwd()
if (ROOT / "ml-engine").is_dir():
    ROOT = ROOT / "ml-engine"
sys.path.insert(0, str(ROOT))
print("ml-engine root:", ROOT)

In [ ]:
from training.dataset import DEFAULT_DATASETS, corpora_present, primary_train_ready
DATASETS = Path("/content/datasets")  # or ../datasets after clone
if not DATASETS.is_dir():
    DATASETS = DEFAULT_DATASETS
print("presence:", corpora_present(DATASETS))
print("train_ready:", primary_train_ready(DATASETS))
SYNTHETIC = not primary_train_ready(DATASETS)
if SYNTHETIC:
    print("WARNING: no ASVspoof on disk — will use --synthetic smoke corpus (labelled).")

In [ ]:
from training.train_antispoof import main as train_main
OUT = ROOT / "models" / "antispoof"
argv = [
    "--datasets", str(DATASETS),
    "--out", str(OUT),
    "--epochs", "10",
    "--batch-size", "32",
    "--device", "auto",
    "--resume",
]
if SYNTHETIC:
    argv += ["--synthetic", "--limit", "200"]
rc = train_main(argv)
assert rc == 0, rc

In [ ]:
from training.calibrate import main as cal_main
ckpt = OUT / "codec_aug.pt"
argv = [
    "--checkpoint", str(ckpt),
    "--datasets", str(DATASETS),
    "--out", str(OUT),
    "--device", "auto",
]
if SYNTHETIC:
    argv += ["--synthetic", "--limit", "200"]
rc = cal_main(argv)
assert rc == 0, rc
print("Wrote", OUT / "calibration.json")

Download models/antispoof/*.pt + calibration.json back to the laptop, then:

`ash
python -m benchmarks.run_eval --datasets ../datasets --seed 42
python -m benchmarks.codec_study --datasets ../datasets --seed 42
`

Without corpora those commands **stop** and leave synthetic report cells labelled.